In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)


def vector_projection_onto_plane(v, n):
    v = np.array(v)
    n = np.array(n)
    return v - np.dot(v, n) * n / np.linalg.norm(n)**2


folder_path = 'D:/CFD/F231'
frames_info_path = os.path.join(folder_path, 'F231_frames.csv')


frames_info = pd.read_csv(frames_info_path)


csv_files = [f for f in os.listdir(folder_path) if f.startswith('slice_') and f.endswith('.csv') and f[6:-4].isdigit()]
csv_files = sorted(csv_files, key=lambda x: int(x.split('_')[1].split('.')[0]))


slice_numbers = [int(f.split('_')[1].split('.')[0]) for f in csv_files]

print(f"Slice 범위: {min(slice_numbers)} ~ {max(slice_numbers)}")


heatmap_data = pd.DataFrame()


for slice_num in slice_numbers:
    file_path = os.path.join(folder_path, f'slice_{slice_num}.csv')
    df = pd.read_csv(file_path)

    
    normal_row = frames_info[frames_info['Frame'] == slice_num]
    if normal_row.empty:
        print(f"Warning: Frame {slice_num} not found in frames_info.")
        continue
    normal_vector = normal_row[['E', 'F', 'G']].values.flatten()


    centroid = df[['Points:0', 'Points:1', 'Points:2']].mean()


    projected_vector = vector_projection_onto_plane([1, 0, 0], normal_vector)
    x_axis = projected_vector / np.linalg.norm(projected_vector)
    z_axis = normal_vector / np.linalg.norm(normal_vector)
    y_axis = np.cross(z_axis, x_axis)

    transformation_matrix = np.column_stack((x_axis, y_axis, z_axis))
    shifted_coords = df[['Points:0', 'Points:1', 'Points:2']] - centroid
    transformed_coords = np.dot(shifted_coords, transformation_matrix)

    df['New_X'] = transformed_coords[:, 0]
    df['New_Y'] = transformed_coords[:, 1]
    df['New_Z'] = transformed_coords[:, 2]


    angles = np.degrees(np.arctan2(df['New_Y'], df['New_X']))
    angles = (angles + 360) % 360
    df['Angle'] = angles


    df['Angle_Bin'] = pd.cut(df['Angle'], bins=np.arange(0, 370, 10), right=False, include_lowest=True)

    avg_shear = df.groupby('Angle_Bin', observed=False)['wall_shear'].mean()


    avg_shear.index = [f"{int(interval.left)}-{int(interval.right)}" for interval in avg_shear.index]
    avg_shear.name = slice_num
    heatmap_data = pd.concat([heatmap_data, avg_shear], axis=1)

heatmap_data = heatmap_data.T
heatmap_data.index.name = 'Slice'


heatmap_data = heatmap_data.interpolate(axis=0).interpolate(axis=1)


plt.figure(figsize=(8, 12))
ax = sns.heatmap(
    heatmap_data,
    cmap='viridis',
    vmin=0, vmax=60,
    cbar_kws={'label': 'Wall Shear Stress (Pa)'}
)

# X축: Angle range
ax.set_xlabel('Angle Range (degrees)', fontsize=14)
ax.set_xticks(np.linspace(0, heatmap_data.shape[1], 5))  
ax.set_xticklabels(['0', '90', '180', '270', '360'], fontsize=12)

# Y축: Slice number
ax.set_ylabel('Slice Number', fontsize=14)
num_slices = heatmap_data.shape[0]
yticks = np.linspace(0, num_slices-1, 10).astype(int)  
ax.set_yticks(yticks)
ax.set_yticklabels(heatmap_data.index[yticks], fontsize=10)

# 제목
plt.title('F231', fontsize=16, pad=20)

# 레이아웃 조정
plt.tight_layout()
plt.show()


ModuleNotFoundError: No module named 'seaborn'